# Prediction Soup

In [ ]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
from collections import Counter
from torch.nn import functional as F
import kagglehub
import time
import logging
import matplotlib.pyplot as plt
from datetime import datetime
import re
import csv

CHECKPOINT_DIR = './Soup_Saved_5_ViTB'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SPLIT_FILE = "split_indices.pkl"

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ClassBalancedLoss(torch.nn.Module):
    def __init__(self, beta, gamma, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, samples_per_cls):
        num_classes = logits.size(1)
        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        weights = weights.to(logits.device)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1 - probs, self.gamma)
        cb_loss = -weights.unsqueeze(0) * focal_weight * targets_one_hot * log_probs
        loss = cb_loss.sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

class Caltech256Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.images = []
        self.labels = []
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, img_name))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": torch.tensor(label, dtype=torch.long)}

def load_dataset(root_dir, transform):
    return Caltech256Dataset(root_dir=root_dir, transform=transform)

def split_dataset(dataset, test_size=0.3, val_size=0.5, random_state=11):
    labels = dataset.labels
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))
    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    val_idx, test_idx = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))
    val_idx = [temp_idx[i] for i in val_idx]
    test_idx = [temp_idx[i] for i in test_idx]
    return train_idx, val_idx, test_idx

def save_split_indices(train_idx, val_idx, test_idx, file_path=SPLIT_FILE):
    with open(file_path, "wb") as f:
        pickle.dump((train_idx, val_idx, test_idx), f)
    print(f"Split indices saved to {file_path}")

def load_split_indices(file_path=SPLIT_FILE):
    with open(file_path, "rb") as f:
        train_idx, val_idx, test_idx = pickle.load(f)
    print(f"Loaded split indices from {file_path}")
    return train_idx, val_idx, test_idx

def get_transform(processor, aug_params=None):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])

def get_model(model_name, num_classes, freeze_layers=0, local_model_path=None):
    if local_model_path and os.path.exists(local_model_path):
        print(f"Loading model from local folder: {local_model_path}")
        model = ViTForImageClassification.from_pretrained(
            local_model_path,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        print("Downloading model from Hugging Face...")
        model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        if local_model_path:
            os.makedirs(local_model_path, exist_ok=True)
            model.save_pretrained(local_model_path)
            print(f"Model saved to {local_model_path}")
    if freeze_layers > 0:
        num_layers = len(model.vit.encoder.layer)
        freeze_layers = min(freeze_layers, num_layers)
        for i in range(freeze_layers):
            for param in model.vit.encoder.layer[i].parameters():
                param.requires_grad = False
        print(f"Froze the first {freeze_layers} layers of the ViT encoder.")
    return model

def get_loss_function(loss_type, **kwargs):
    if loss_type == 'focal':
        return FocalLoss(gamma=kwargs.get('gamma', 2.0))
    elif loss_type == 'class_balanced':
        return ClassBalancedLoss(beta=kwargs.get('beta', 0.9999), gamma=kwargs.get('gamma', 2.0))
    else:
        raise ValueError("unsupported loss type")

def evaluate_model(model, data_loader, criterion, device, samples_per_cls=None):
    model.eval()
    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            if labels_batch.max() >= model.classifier.out_features:
                raise ValueError(f"Label {labels_batch.max().item()} exceeds model classes {model.classifier.out_features}")
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values)
                logits = outputs.logits
                loss = criterion(logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    return avg_loss, accuracy, macro_f1
def perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    uniform_soup_state_dict = average_state_dicts(state_dicts)
    
    model.load_state_dict(uniform_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'uniform_soup.pth')
    torch.save({'model_state_dict': uniform_soup_state_dict}, soup_checkpoint_path)
    print(f"Uniform soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Uniform Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    accuracies = []
    for i, state_dict in enumerate(state_dicts):
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        accuracies.append(acc)
        model.to('cpu')
        torch.cuda.empty_cache()

    weights = torch.softmax(torch.tensor(accuracies), dim=0).tolist()
    weighted_soup_state_dict = {key: sum(w * sd[key] for w, sd in zip(weights, state_dicts)) for key in state_dicts[0].keys()}

    model.load_state_dict(weighted_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'weighted_soup.pth')
    torch.save({'model_state_dict': weighted_soup_state_dict}, soup_checkpoint_path)
    print(f"Weighted soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu') 
    torch.cuda.empty_cache() 
    print(f"Weighted Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def compute_cosine_similarity(state_dict1, state_dict2):
    flat1 = torch.cat([p.flatten() for p in state_dict1.values()])
    flat2 = torch.cat([p.flatten() for p in state_dict2.values()])
    return F.cosine_similarity(flat1, flat2, dim=0).item()

def perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config, k=3):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    selected_indices = [0]
    for _ in range(k - 1):
        max_diversity = -1
        best_idx = None
        for i in range(len(state_dicts)):
            if i not in selected_indices:
                avg_similarity = sum(compute_cosine_similarity(state_dicts[i], state_dicts[j]) for j in selected_indices) / len(selected_indices)
                diversity = 1 - avg_similarity
                if diversity > max_diversity:
                    max_diversity = diversity
                    best_idx = i
        selected_indices.append(best_idx)

    diverse_state_dicts = [state_dicts[i] for i in selected_indices]
    diversity_soup_state_dict = average_state_dicts(diverse_state_dicts)
    
    model.load_state_dict(diversity_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'diversity_soup.pth')
    torch.save({'model_state_dict': diversity_soup_state_dict}, soup_checkpoint_path)
    print(f"Diversity soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu') 
    torch.cuda.empty_cache() 
    print(f"Diversity Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    print(f"Model paths to process: {model_paths}")
    
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)

    state_dict_hashes = []
    for i, path in enumerate(model_paths):
        state_dict = torch.load(path, map_location='cpu')['model_state_dict']
        flat_weights = torch.cat([p.flatten() for p in state_dict.values()])
        hash_value = hash(flat_weights.cpu().numpy().tobytes())
        state_dict_hashes.append((path, hash_value))
        print(f"Model {i+1}/{len(model_paths)} at {path} has hash: {hash_value}")
    
    unique_hashes = len(set(h[1] for h in state_dict_hashes))
    if unique_hashes < len(model_paths):
        print(f"WARNING: Only {unique_hashes} unique models detected out of {len(model_paths)} files!")

    total_loss = 0
    preds = []
    true_labels = []
    
    total_batches = len(test_loader)
    print(f"Starting Prediction Soup with {total_batches} batches and {len(model_paths)} models.")
    start_time = time.time()

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            batch_start_time = time.time()
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            avg_logits = torch.zeros(len(labels_batch), num_classes, device=device)
            
            for i, path in enumerate(model_paths):
                state_dict = torch.load(path, map_location='cpu')['model_state_dict']
                checkpoint_classes = state_dict['classifier.weight'].shape[0]
                if checkpoint_classes != num_classes:
                    print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
                    if checkpoint_classes < num_classes:
                        padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                        padding_bias = torch.zeros(num_classes - checkpoint_classes)
                        state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                        state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
                    elif checkpoint_classes > num_classes:
                        state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                        state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
                try:
                    model.load_state_dict(state_dict)
                except Exception as e:
                    print(f"Error loading state_dict from {path}: {e}")
                    continue
                model.to(device)
                model.eval()
                outputs = model(pixel_values)
                if batch_idx == 0: 
                    print(f"Model {i+1}/{len(model_paths)} from {path} - Sample logits: {outputs.logits[0][:5]}")
                avg_logits += outputs.logits / len(model_paths)
                model.to('cpu')
                torch.cuda.empty_cache()
                print(f"  Processed model {i+1}/{len(model_paths)} from {path}")

            loss = criterion(avg_logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(avg_logits, labels_batch)
            total_loss += loss.item()
            preds.extend(avg_logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

            batches_completed = batch_idx + 1
            batches_remaining = total_batches - batches_completed
            elapsed_time = time.time() - start_time
            batch_time = time.time() - batch_start_time
            avg_time_per_batch = elapsed_time / batches_completed
            estimated_time_remaining = avg_time_per_batch * batches_remaining
            
            print(f"Batch {batches_completed}/{total_batches} completed:")
            print(f"  Time for this batch: {batch_time:.2f} seconds")
            print(f"  Batches remaining: {batches_remaining}")
            print(f"  Elapsed time: {elapsed_time:.2f} seconds")
            print(f"  Estimated time remaining: {estimated_time_remaining:.2f} seconds")

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    
    total_time = time.time() - start_time
    print(f"\nPrediction Soup Completed:")
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Average time per batch: {total_time / total_batches:.2f} seconds")
    print(f"Test Results - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Macro F1: {macro_f1:.4f}")


def perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    cpu_device = torch.device('cpu')
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']
    
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    
    state_dicts = []
    for path in model_paths:
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint['model_state_dict']
        checkpoint_classes = state_dict['classifier.weight'].shape[0]
        if checkpoint_classes != num_classes:
            print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
            if checkpoint_classes < num_classes:
                padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                padding_bias = torch.zeros(num_classes - checkpoint_classes)
                state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
            elif checkpoint_classes > num_classes:
                state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
        state_dicts.append(state_dict)
    
    individual_accuracies = []
    for i, state_dict in enumerate(state_dicts):
        print(f"Loading and evaluating model {i} from {model_paths[i]}")
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        individual_accuracies.append((i, acc))
        print(f"Model {i}: Validation Accuracy = {acc:.4f}")
        model.to(cpu_device)
        torch.cuda.empty_cache()
    
    sorted_models = sorted(individual_accuracies, key=lambda x: x[1], reverse=True)
    print("\nSorted models by validation accuracy:")
    for idx, acc in sorted_models:
        print(f"Model {idx}: {acc:.4f}")
    
    best_idx, best_acc = sorted_models[0]
    soup_indices = [best_idx]
    current_soup_acc = best_acc
    print(f"\nInitial soup with model {best_idx}, accuracy: {best_acc:.4f}")
    
    for idx, acc in sorted_models[1:]:
        temp_soup_indices = soup_indices + [idx]
        temp_soup_state_dict = average_state_dicts([state_dicts[i] for i in temp_soup_indices])
        model.load_state_dict(temp_soup_state_dict)
        model.to(device) 
        print(f"Evaluating temporary soup with models {temp_soup_indices}")
        _, temp_acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        model.to(cpu_device)
        torch.cuda.empty_cache()
        if temp_acc > current_soup_acc:
            soup_indices.append(idx)
            current_soup_acc = temp_acc
            print(f"Added model {idx}, new soup accuracy: {temp_acc:.4f}")
        else:
            print(f"Model {idx} did not improve soup, skipping (accuracy: {temp_acc:.4f})")
    
    final_soup_state_dict = average_state_dicts([state_dicts[i] for i in soup_indices])
    model.load_state_dict(final_soup_state_dict)
    model.to(device)  
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'greedy_soup.pth')
    torch.save({'model_state_dict': final_soup_state_dict}, soup_checkpoint_path)
    print(f"Final soup saved to {soup_checkpoint_path}")
    
    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to(cpu_device) 
    torch.cuda.empty_cache() 
    print(f"\nFinal Soup Test Results:")
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")

def average_state_dicts(state_dicts):
    if not state_dicts:
        raise ValueError("No state_dicts to average")
    soup_state_dict = {}
    for key in state_dicts[0].keys():
        soup_state_dict[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return soup_state_dict

def main(config):
    torch.manual_seed(config['seed'])
    np.random.seed(config['seed'])
    torch.backends.cudnn.deterministic = True

    if os.path.exists(config['model_local_path']):
        print(f"Loading processor from local folder: {config['model_local_path']}")
        processor = ViTImageProcessor.from_pretrained(config['model_local_path'])
    else:
        print("Downloading processor from Hugging Face...")
        processor = ViTImageProcessor.from_pretrained(config['model_name'])
        os.makedirs(config['model_local_path'], exist_ok=True)
        processor.save_pretrained(config['model_local_path'])
        print(f"Processor saved to {config['model_local_path']}")

    transform = get_transform(processor)
    dataset = load_dataset(config['dataset_path'], transform)
    
    print(f"Number of classes in dataset: {len(dataset.classes)}")
    print("Class names:")
    for i, cls_name in enumerate(dataset.classes):
        print(f"Class {i}: {cls_name}")
    print(f"Min label: {min(dataset.labels)}, Max label: {max(dataset.labels)}")
    if max(dataset.labels) >= config['num_classes']:
        print(f"Warning: Dataset has labels up to {max(dataset.labels)}, but model expects {config['num_classes']} classes")
        config['num_classes'] = len(dataset.classes)
        print(f"Adjusted config['num_classes'] to {config['num_classes']} to match dataset")

    if os.path.exists(SPLIT_FILE):
        train_idx, val_idx, test_idx = load_split_indices(SPLIT_FILE)
    else:
        train_idx, val_idx, test_idx = split_dataset(dataset, config['test_size'], config['val_size'], config['seed'])
        save_split_indices(train_idx, val_idx, test_idx, SPLIT_FILE)

    val_dataset = torch.utils.data.Subset(dataset, val_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

    if config['loss_type'] == 'class_balanced':
        train_labels = [dataset.labels[i] for i in train_idx]
        samples_per_cls = torch.zeros(config['num_classes'])
        for i, count in Counter(train_labels).items():
            samples_per_cls[i] = count
        criterion = get_loss_function(config['loss_type'], beta=config['beta'], gamma=config['gamma'])
    else:
        criterion = get_loss_function(config['loss_type'], gamma=config['gamma'])
        samples_per_cls = None

    soup_dir = 'ViT-B_Soup'
    model_paths = [os.path.join(soup_dir, f) for f in os.listdir(soup_dir) if f.endswith('.pth')]
    if model_paths:
        print(f"Found {len(model_paths)} models in {soup_dir}: {model_paths}")
        if config['soup_type'] == 'greedy':
            perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'uniform':
            perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'weighted':
            perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'diversity':
            perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'prediction':
            perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config)
    else:
        print(f"No .pth files found in {soup_dir}. Skipping greedy soup.")

config = {
    'seed': 10,
    'model_name': "google/vit-base-patch16-224-in21k",
    'model_local_path': "./local_model/vit-base-patch16-224-in21k",
    'num_classes': 257,
    'dataset_path': os.path.join(kagglehub.dataset_download("jessicali9530/caltech256"), "256_ObjectCategories"),
    'test_size': 0.3,
    'val_size': 0.5,
    'augmentation_params': {},
    'batch_size': 128,
    'num_workers': 0,
    'lr': 1e-4,
    'T_max': 25,
    'soup_type':'prediction',
    'loss_type': 'focal',
    'beta': 0.9999,
    'gamma': 2.0,
    'num_epochs': 25,
    'patience': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'load_checkpoint': True,
    'freeze_layers': 6,
}

if __name__ == "__main__":
    main(config)

c:\Users\Dell-G5\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading processor from local folder: ./local_model/vit-base-patch16-224-in21k
Number of classes in dataset: 257
Class names:
Class 0: 001.ak47
Class 1: 002.american-flag
Class 2: 003.backpack
Class 3: 004.baseball-bat
Class 4: 005.baseball-glove
Class 5: 006.basketball-hoop
Class 6: 007.bat
Class 7: 008.bathtub
Class 8: 009.bear
Class 9: 010.beer-mug
Class 10: 011.billiards
Class 11: 012.binoculars
Class 12: 013.birdbath
Class 13: 014.blimp
Class 14: 015.bonsai-101
Class 15: 016.boom-box
Class 16: 017.bowling-ball
Class 17: 018.bowling-pin
Class 18: 019.boxing-glove
Class 19: 020.brain-101
Class 20: 021.breadmaker
Class 21: 022.buddha-101
Class 22: 023.bulldozer
Class 23: 024.butterfly
Class 24: 025.cactus
Class 25: 026.cake
Class 26: 027.calculator
Class 27: 028.camel
Class 28: 029.cannon
Class 29: 030.canoe
Class 30: 031.car-tire
Class 31: 032.cartman
Class 32: 033.cd
Class 33: 034.centipede
Class 34: 035.cereal-box
Class 35: 036.chandelier-101
Class 36: 037.chess-board
Class 37: 038

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./local_model/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([257]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([257, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model 1/5 at ViT-B_Soup\ViT-B1.pth has hash: -4339046433408528246
Model 2/5 at ViT-B_Soup\ViT-B2.pth has hash: -7794475835567905912
Model 3/5 at ViT-B_Soup\ViT-B3.pth has hash: -2994255624981955264
Model 4/5 at ViT-B_Soup\ViT-B4.pth has hash: -618267326999065796
Model 5/5 at ViT-B_Soup\ViT-B5.pth has hash: 5122666344876974622
Starting Prediction Soup with 36 batches and 5 models.
Model 1/5 from ViT-B_Soup\ViT-B1.pth - Sample logits: tensor([ 0.4796, -0.5922,  0.1490,  0.0854, -0.4982], device='cuda:0')
  Processed model 1/5 from ViT-B_Soup\ViT-B1.pth
Model 2/5 from ViT-B_Soup\ViT-B2.pth - Sample logits: tensor([ 0.2841, -0.0946,  0.1228, -0.1325, -0.1757], device='cuda:0')
  Processed model 2/5 from ViT-B_Soup\ViT-B2.pth
Model 3/5 from ViT-B_Soup\ViT-B3.pth - Sample logits: tensor([ 0.3263, -0.2750, -0.0126, -0.1182, -0.4052], device='cuda:0')
  Processed model 3/5 from ViT-B_Soup\ViT-B3.pth
Model 4/5 from ViT-B_Soup\ViT-B4.pth - Sample logits: tensor([ 0.2996, -0.2741,  0.1394,  0.263

# Diversity Soup

In [ ]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
from collections import Counter
from torch.nn import functional as F
import kagglehub
import time
import logging
import matplotlib.pyplot as plt
from datetime import datetime
import re
import csv

CHECKPOINT_DIR = './Soup_Saved_5_ViTB'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SPLIT_FILE = "split_indices.pkl"

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ClassBalancedLoss(torch.nn.Module):
    def __init__(self, beta, gamma, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, samples_per_cls):
        num_classes = logits.size(1)
        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        weights = weights.to(logits.device)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1 - probs, self.gamma)
        cb_loss = -weights.unsqueeze(0) * focal_weight * targets_one_hot * log_probs
        loss = cb_loss.sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

class Caltech256Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.images = []
        self.labels = []
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, img_name))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": torch.tensor(label, dtype=torch.long)}

def load_dataset(root_dir, transform):
    return Caltech256Dataset(root_dir=root_dir, transform=transform)

def split_dataset(dataset, test_size=0.3, val_size=0.5, random_state=11):
    labels = dataset.labels
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))
    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    val_idx, test_idx = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))
    val_idx = [temp_idx[i] for i in val_idx]
    test_idx = [temp_idx[i] for i in test_idx]
    return train_idx, val_idx, test_idx

def save_split_indices(train_idx, val_idx, test_idx, file_path=SPLIT_FILE):
    with open(file_path, "wb") as f:
        pickle.dump((train_idx, val_idx, test_idx), f)
    print(f"Split indices saved to {file_path}")

def load_split_indices(file_path=SPLIT_FILE):
    with open(file_path, "rb") as f:
        train_idx, val_idx, test_idx = pickle.load(f)
    print(f"Loaded split indices from {file_path}")
    return train_idx, val_idx, test_idx

def get_transform(processor, aug_params=None):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])

def get_model(model_name, num_classes, freeze_layers=0, local_model_path=None):
    if local_model_path and os.path.exists(local_model_path):
        print(f"Loading model from local folder: {local_model_path}")
        model = ViTForImageClassification.from_pretrained(
            local_model_path,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        print("Downloading model from Hugging Face...")
        model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        if local_model_path:
            os.makedirs(local_model_path, exist_ok=True)
            model.save_pretrained(local_model_path)
            print(f"Model saved to {local_model_path}")
    if freeze_layers > 0:
        num_layers = len(model.vit.encoder.layer)
        freeze_layers = min(freeze_layers, num_layers)
        for i in range(freeze_layers):
            for param in model.vit.encoder.layer[i].parameters():
                param.requires_grad = False
        print(f"Froze the first {freeze_layers} layers of the ViT encoder.")
    return model

def get_loss_function(loss_type, **kwargs):
    if loss_type == 'focal':
        return FocalLoss(gamma=kwargs.get('gamma', 2.0))
    elif loss_type == 'class_balanced':
        return ClassBalancedLoss(beta=kwargs.get('beta', 0.9999), gamma=kwargs.get('gamma', 2.0))
    else:
        raise ValueError("unsupported loss type")

def evaluate_model(model, data_loader, criterion, device, samples_per_cls=None):
    model.eval()
    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            if labels_batch.max() >= model.classifier.out_features:
                raise ValueError(f"Label {labels_batch.max().item()} exceeds model classes {model.classifier.out_features}")
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values)
                logits = outputs.logits
                loss = criterion(logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    return avg_loss, accuracy, macro_f1
def perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    uniform_soup_state_dict = average_state_dicts(state_dicts)
    
    model.load_state_dict(uniform_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'uniform_soup.pth')
    torch.save({'model_state_dict': uniform_soup_state_dict}, soup_checkpoint_path)
    print(f"Uniform soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu') 
    torch.cuda.empty_cache()  
    print(f"Uniform Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    accuracies = []
    for i, state_dict in enumerate(state_dicts):
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        accuracies.append(acc)
        model.to('cpu')
        torch.cuda.empty_cache()

    weights = torch.softmax(torch.tensor(accuracies), dim=0).tolist()
    weighted_soup_state_dict = {key: sum(w * sd[key] for w, sd in zip(weights, state_dicts)) for key in state_dicts[0].keys()}

    model.load_state_dict(weighted_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'weighted_soup.pth')
    torch.save({'model_state_dict': weighted_soup_state_dict}, soup_checkpoint_path)
    print(f"Weighted soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')
    torch.cuda.empty_cache()
    print(f"Weighted Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def compute_cosine_similarity(state_dict1, state_dict2):
    flat1 = torch.cat([p.flatten() for p in state_dict1.values()])
    flat2 = torch.cat([p.flatten() for p in state_dict2.values()])
    return F.cosine_similarity(flat1, flat2, dim=0).item()

def perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config, k=3):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    selected_indices = [0]
    for _ in range(k - 1):
        max_diversity = -1
        best_idx = None
        for i in range(len(state_dicts)):
            if i not in selected_indices:
                avg_similarity = sum(compute_cosine_similarity(state_dicts[i], state_dicts[j]) for j in selected_indices) / len(selected_indices)
                diversity = 1 - avg_similarity
                if diversity > max_diversity:
                    max_diversity = diversity
                    best_idx = i
        selected_indices.append(best_idx)

    diverse_state_dicts = [state_dicts[i] for i in selected_indices]
    diversity_soup_state_dict = average_state_dicts(diverse_state_dicts)
    
    model.load_state_dict(diversity_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'diversity_soup.pth')
    torch.save({'model_state_dict': diversity_soup_state_dict}, soup_checkpoint_path)
    print(f"Diversity soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')
    torch.cuda.empty_cache()
    print(f"Diversity Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            avg_logits = torch.zeros(len(labels_batch), num_classes, device=device)
            for path in model_paths:
                model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
                state_dict = torch.load(path, map_location='cpu')['model_state_dict']
                model.load_state_dict(state_dict)
                model.to(device)
                model.eval()
                outputs = model(pixel_values)
                avg_logits += outputs.logits / len(model_paths)
                model.to('cpu')
                torch.cuda.empty_cache()
            loss = criterion(avg_logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(avg_logits, labels_batch)
            total_loss += loss.item()
            preds.extend(avg_logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    print(f"Prediction Soup Test Results: Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Macro F1: {macro_f1:.4f}")


def perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    cpu_device = torch.device('cpu')
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']
    
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    
    state_dicts = []
    for path in model_paths:
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint['model_state_dict']
        checkpoint_classes = state_dict['classifier.weight'].shape[0]
        if checkpoint_classes != num_classes:
            print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
            if checkpoint_classes < num_classes:
                padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                padding_bias = torch.zeros(num_classes - checkpoint_classes)
                state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
            elif checkpoint_classes > num_classes:
                state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
        state_dicts.append(state_dict)
    
    individual_accuracies = []
    for i, state_dict in enumerate(state_dicts):
        print(f"Loading and evaluating model {i} from {model_paths[i]}")
        model.load_state_dict(state_dict)
        model.to(device)  
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        individual_accuracies.append((i, acc))
        print(f"Model {i}: Validation Accuracy = {acc:.4f}")
        model.to(cpu_device) 
        torch.cuda.empty_cache()
    
    sorted_models = sorted(individual_accuracies, key=lambda x: x[1], reverse=True)
    print("\nSorted models by validation accuracy:")
    for idx, acc in sorted_models:
        print(f"Model {idx}: {acc:.4f}")
    
    best_idx, best_acc = sorted_models[0]
    soup_indices = [best_idx]
    current_soup_acc = best_acc
    print(f"\nInitial soup with model {best_idx}, accuracy: {best_acc:.4f}")
    
    for idx, acc in sorted_models[1:]:
        temp_soup_indices = soup_indices + [idx]
        temp_soup_state_dict = average_state_dicts([state_dicts[i] for i in temp_soup_indices])
        model.load_state_dict(temp_soup_state_dict)
        model.to(device)  
        print(f"Evaluating temporary soup with models {temp_soup_indices}")
        _, temp_acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        model.to(cpu_device) 
        torch.cuda.empty_cache() 
        if temp_acc > current_soup_acc:
            soup_indices.append(idx)
            current_soup_acc = temp_acc
            print(f"Added model {idx}, new soup accuracy: {temp_acc:.4f}")
        else:
            print(f"Model {idx} did not improve soup, skipping (accuracy: {temp_acc:.4f})")
    
    final_soup_state_dict = average_state_dicts([state_dicts[i] for i in soup_indices])
    model.load_state_dict(final_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'greedy_soup.pth')
    torch.save({'model_state_dict': final_soup_state_dict}, soup_checkpoint_path)
    print(f"Final soup saved to {soup_checkpoint_path}")
    
    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to(cpu_device)
    torch.cuda.empty_cache()
    print(f"\nFinal Soup Test Results:")
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")

def average_state_dicts(state_dicts):
    if not state_dicts:
        raise ValueError("No state_dicts to average")
    soup_state_dict = {}
    for key in state_dicts[0].keys():
        soup_state_dict[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return soup_state_dict

def main(config):
    torch.manual_seed(config['seed'])
    np.random.seed(config['seed'])
    torch.backends.cudnn.deterministic = True

    if os.path.exists(config['model_local_path']):
        print(f"Loading processor from local folder: {config['model_local_path']}")
        processor = ViTImageProcessor.from_pretrained(config['model_local_path'])
    else:
        print("Downloading processor from Hugging Face...")
        processor = ViTImageProcessor.from_pretrained(config['model_name'])
        os.makedirs(config['model_local_path'], exist_ok=True)
        processor.save_pretrained(config['model_local_path'])
        print(f"Processor saved to {config['model_local_path']}")

    transform = get_transform(processor)
    dataset = load_dataset(config['dataset_path'], transform)
    
    print(f"Number of classes in dataset: {len(dataset.classes)}")
    print("Class names:")
    for i, cls_name in enumerate(dataset.classes):
        print(f"Class {i}: {cls_name}")
    print(f"Min label: {min(dataset.labels)}, Max label: {max(dataset.labels)}")
    if max(dataset.labels) >= config['num_classes']:
        print(f"Warning: Dataset has labels up to {max(dataset.labels)}, but model expects {config['num_classes']} classes")
        config['num_classes'] = len(dataset.classes)
        print(f"Adjusted config['num_classes'] to {config['num_classes']} to match dataset")

    if os.path.exists(SPLIT_FILE):
        train_idx, val_idx, test_idx = load_split_indices(SPLIT_FILE)
    else:
        train_idx, val_idx, test_idx = split_dataset(dataset, config['test_size'], config['val_size'], config['seed'])
        save_split_indices(train_idx, val_idx, test_idx, SPLIT_FILE)

    val_dataset = torch.utils.data.Subset(dataset, val_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

    if config['loss_type'] == 'class_balanced':
        train_labels = [dataset.labels[i] for i in train_idx]
        samples_per_cls = torch.zeros(config['num_classes'])
        for i, count in Counter(train_labels).items():
            samples_per_cls[i] = count
        criterion = get_loss_function(config['loss_type'], beta=config['beta'], gamma=config['gamma'])
    else:
        criterion = get_loss_function(config['loss_type'], gamma=config['gamma'])
        samples_per_cls = None

    soup_dir = 'ViT-B_Soup'
    model_paths = [os.path.join(soup_dir, f) for f in os.listdir(soup_dir) if f.endswith('.pth')]
    if model_paths:
        print(f"Found {len(model_paths)} models in {soup_dir}: {model_paths}")
        if config['soup_type'] == 'greedy':
            perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'uniform':
            perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'weighted':
            perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'diversity':
            perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'prediction':
            perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config)
    else:
        print(f"No .pth files found in {soup_dir}. Skipping greedy soup.")

config = {
    'seed': 10,
    'model_name': "google/vit-base-patch16-224-in21k",
    'model_local_path': "./local_model/vit-base-patch16-224-in21k",
    'num_classes': 257,
    'dataset_path': os.path.join(kagglehub.dataset_download("jessicali9530/caltech256"), "256_ObjectCategories"),
    'test_size': 0.3,
    'val_size': 0.5,
    'augmentation_params': {},
    'batch_size': 64,
    'num_workers': 0,
    'lr': 1e-4,
    'T_max': 25,
    'soup_type':'diversity',# soups: greedy, diversity, uniform, weighted ; Ensemble Learning: prediction
    'loss_type': 'focal',
    'beta': 0.9999,
    'gamma': 2.0,
    'num_epochs': 25,
    'patience': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'load_checkpoint': True,
    'freeze_layers': 6,
}

if __name__ == "__main__":
    main(config)

Loading processor from local folder: ./local_model/vit-base-patch16-224-in21k
Number of classes in dataset: 257
Class names:
Class 0: 001.ak47
Class 1: 002.american-flag
Class 2: 003.backpack
Class 3: 004.baseball-bat
Class 4: 005.baseball-glove
Class 5: 006.basketball-hoop
Class 6: 007.bat
Class 7: 008.bathtub
Class 8: 009.bear
Class 9: 010.beer-mug
Class 10: 011.billiards
Class 11: 012.binoculars
Class 12: 013.birdbath
Class 13: 014.blimp
Class 14: 015.bonsai-101
Class 15: 016.boom-box
Class 16: 017.bowling-ball
Class 17: 018.bowling-pin
Class 18: 019.boxing-glove
Class 19: 020.brain-101
Class 20: 021.breadmaker
Class 21: 022.buddha-101
Class 22: 023.bulldozer
Class 23: 024.butterfly
Class 24: 025.cactus
Class 25: 026.cake
Class 26: 027.calculator
Class 27: 028.camel
Class 28: 029.cannon
Class 29: 030.canoe
Class 30: 031.car-tire
Class 31: 032.cartman
Class 32: 033.cd
Class 33: 034.centipede
Class 34: 035.cereal-box
Class 35: 036.chandelier-101
Class 36: 037.chess-board
Class 37: 038

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./local_model/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([257]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([257, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Diversity soup saved to ./Soup_Saved_5_ViTB\diversity_soup.pth
Diversity Soup Test Results: Loss: 1.0904, Accuracy: 0.9312, Macro F1: 0.9335


# Weighted Soup

In [ ]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
from collections import Counter
from torch.nn import functional as F
import kagglehub
import time
import logging
import matplotlib.pyplot as plt
from datetime import datetime
import re
import csv

CHECKPOINT_DIR = './Soup_Saved_5_ViTB'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SPLIT_FILE = "split_indices.pkl"

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ClassBalancedLoss(torch.nn.Module):
    def __init__(self, beta, gamma, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, samples_per_cls):
        num_classes = logits.size(1)
        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        weights = weights.to(logits.device)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1 - probs, self.gamma)
        cb_loss = -weights.unsqueeze(0) * focal_weight * targets_one_hot * log_probs
        loss = cb_loss.sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

class Caltech256Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.images = []
        self.labels = []
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, img_name))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": torch.tensor(label, dtype=torch.long)}

def load_dataset(root_dir, transform):
    return Caltech256Dataset(root_dir=root_dir, transform=transform)

def split_dataset(dataset, test_size=0.3, val_size=0.5, random_state=11):
    labels = dataset.labels
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))
    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    val_idx, test_idx = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))
    val_idx = [temp_idx[i] for i in val_idx]
    test_idx = [temp_idx[i] for i in test_idx]
    return train_idx, val_idx, test_idx

def save_split_indices(train_idx, val_idx, test_idx, file_path=SPLIT_FILE):
    with open(file_path, "wb") as f:
        pickle.dump((train_idx, val_idx, test_idx), f)
    print(f"Split indices saved to {file_path}")

def load_split_indices(file_path=SPLIT_FILE):
    with open(file_path, "rb") as f:
        train_idx, val_idx, test_idx = pickle.load(f)
    print(f"Loaded split indices from {file_path}")
    return train_idx, val_idx, test_idx

def get_transform(processor, aug_params=None):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])

def get_model(model_name, num_classes, freeze_layers=0, local_model_path=None):
    if local_model_path and os.path.exists(local_model_path):
        print(f"Loading model from local folder: {local_model_path}")
        model = ViTForImageClassification.from_pretrained(
            local_model_path,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        print("Downloading model from Hugging Face...")
        model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        if local_model_path:
            os.makedirs(local_model_path, exist_ok=True)
            model.save_pretrained(local_model_path)
            print(f"Model saved to {local_model_path}")
    if freeze_layers > 0:
        num_layers = len(model.vit.encoder.layer)
        freeze_layers = min(freeze_layers, num_layers)
        for i in range(freeze_layers):
            for param in model.vit.encoder.layer[i].parameters():
                param.requires_grad = False
        print(f"Froze the first {freeze_layers} layers of the ViT encoder.")
    return model

def get_loss_function(loss_type, **kwargs):
    if loss_type == 'focal':
        return FocalLoss(gamma=kwargs.get('gamma', 2.0))
    elif loss_type == 'class_balanced':
        return ClassBalancedLoss(beta=kwargs.get('beta', 0.9999), gamma=kwargs.get('gamma', 2.0))
    else:
        raise ValueError("unsupported loss type")

def evaluate_model(model, data_loader, criterion, device, samples_per_cls=None):
    model.eval()
    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            if labels_batch.max() >= model.classifier.out_features:
                raise ValueError(f"Label {labels_batch.max().item()} exceeds model classes {model.classifier.out_features}")
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values)
                logits = outputs.logits
                loss = criterion(logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    return avg_loss, accuracy, macro_f1
def perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    uniform_soup_state_dict = average_state_dicts(state_dicts)
    
    model.load_state_dict(uniform_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'uniform_soup.pth')
    torch.save({'model_state_dict': uniform_soup_state_dict}, soup_checkpoint_path)
    print(f"Uniform soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')  
    torch.cuda.empty_cache()  
    print(f"Uniform Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    accuracies = []
    for i, state_dict in enumerate(state_dicts):
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        accuracies.append(acc)
        model.to('cpu')
        torch.cuda.empty_cache()

    weights = torch.softmax(torch.tensor(accuracies), dim=0).tolist()
    weighted_soup_state_dict = {key: sum(w * sd[key] for w, sd in zip(weights, state_dicts)) for key in state_dicts[0].keys()}

    model.load_state_dict(weighted_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'weighted_soup.pth')
    torch.save({'model_state_dict': weighted_soup_state_dict}, soup_checkpoint_path)
    print(f"Weighted soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')  
    torch.cuda.empty_cache()
    print(f"Weighted Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def compute_cosine_similarity(state_dict1, state_dict2):
    flat1 = torch.cat([p.flatten() for p in state_dict1.values()])
    flat2 = torch.cat([p.flatten() for p in state_dict2.values()])
    return F.cosine_similarity(flat1, flat2, dim=0).item()

def perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config, k=3):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    selected_indices = [0]
    for _ in range(k - 1):
        max_diversity = -1
        best_idx = None
        for i in range(len(state_dicts)):
            if i not in selected_indices:
                avg_similarity = sum(compute_cosine_similarity(state_dicts[i], state_dicts[j]) for j in selected_indices) / len(selected_indices)
                diversity = 1 - avg_similarity
                if diversity > max_diversity:
                    max_diversity = diversity
                    best_idx = i
        selected_indices.append(best_idx)

    diverse_state_dicts = [state_dicts[i] for i in selected_indices]
    diversity_soup_state_dict = average_state_dicts(diverse_state_dicts)
    
    model.load_state_dict(diversity_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'diversity_soup.pth')
    torch.save({'model_state_dict': diversity_soup_state_dict}, soup_checkpoint_path)
    print(f"Diversity soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')  
    torch.cuda.empty_cache()  
    print(f"Diversity Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            avg_logits = torch.zeros(len(labels_batch), num_classes, device=device)
            for path in model_paths:
                model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
                state_dict = torch.load(path, map_location='cpu')['model_state_dict']
                model.load_state_dict(state_dict)
                model.to(device)
                model.eval()
                outputs = model(pixel_values)
                avg_logits += outputs.logits / len(model_paths)
                model.to('cpu')  
                torch.cuda.empty_cache()  
            loss = criterion(avg_logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(avg_logits, labels_batch)
            total_loss += loss.item()
            preds.extend(avg_logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    print(f"Prediction Soup Test Results: Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Macro F1: {macro_f1:.4f}")


def perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    cpu_device = torch.device('cpu')
    num_classes = config['num_classes']  
    model_name = config['model_name']
    local_model_path = config['model_local_path']
    
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    
    state_dicts = []
    for path in model_paths:
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint['model_state_dict']
        checkpoint_classes = state_dict['classifier.weight'].shape[0]
        if checkpoint_classes != num_classes:
            print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
            if checkpoint_classes < num_classes:
                padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                padding_bias = torch.zeros(num_classes - checkpoint_classes)
                state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
            elif checkpoint_classes > num_classes:
                state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
        state_dicts.append(state_dict)
    
    individual_accuracies = []
    for i, state_dict in enumerate(state_dicts):
        print(f"Loading and evaluating model {i} from {model_paths[i]}")
        model.load_state_dict(state_dict)
        model.to(device)  
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        individual_accuracies.append((i, acc))
        print(f"Model {i}: Validation Accuracy = {acc:.4f}")
        model.to(cpu_device)  
        torch.cuda.empty_cache() 
    
    sorted_models = sorted(individual_accuracies, key=lambda x: x[1], reverse=True)
    print("\nSorted models by validation accuracy:")
    for idx, acc in sorted_models:
        print(f"Model {idx}: {acc:.4f}")
    
    best_idx, best_acc = sorted_models[0]
    soup_indices = [best_idx]
    current_soup_acc = best_acc
    print(f"\nInitial soup with model {best_idx}, accuracy: {best_acc:.4f}")
    
    for idx, acc in sorted_models[1:]:
        temp_soup_indices = soup_indices + [idx]
        temp_soup_state_dict = average_state_dicts([state_dicts[i] for i in temp_soup_indices])
        model.load_state_dict(temp_soup_state_dict)
        model.to(device)  
        print(f"Evaluating temporary soup with models {temp_soup_indices}")
        _, temp_acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        model.to(cpu_device)  
        torch.cuda.empty_cache()  
        if temp_acc > current_soup_acc:
            soup_indices.append(idx)
            current_soup_acc = temp_acc
            print(f"Added model {idx}, new soup accuracy: {temp_acc:.4f}")
        else:
            print(f"Model {idx} did not improve soup, skipping (accuracy: {temp_acc:.4f})")
    
    final_soup_state_dict = average_state_dicts([state_dicts[i] for i in soup_indices])
    model.load_state_dict(final_soup_state_dict)
    model.to(device)  
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'greedy_soup.pth')
    torch.save({'model_state_dict': final_soup_state_dict}, soup_checkpoint_path)
    print(f"Final soup saved to {soup_checkpoint_path}")
    
    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to(cpu_device)  
    torch.cuda.empty_cache()  
    print(f"\nFinal Soup Test Results:")
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")

def average_state_dicts(state_dicts):
    if not state_dicts:
        raise ValueError("No state_dicts to average")
    soup_state_dict = {}
    for key in state_dicts[0].keys():
        soup_state_dict[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return soup_state_dict

def main(config):
    torch.manual_seed(config['seed'])
    np.random.seed(config['seed'])
    torch.backends.cudnn.deterministic = True

    if os.path.exists(config['model_local_path']):
        print(f"Loading processor from local folder: {config['model_local_path']}")
        processor = ViTImageProcessor.from_pretrained(config['model_local_path'])
    else:
        print("Downloading processor from Hugging Face...")
        processor = ViTImageProcessor.from_pretrained(config['model_name'])
        os.makedirs(config['model_local_path'], exist_ok=True)
        processor.save_pretrained(config['model_local_path'])
        print(f"Processor saved to {config['model_local_path']}")

    transform = get_transform(processor)
    dataset = load_dataset(config['dataset_path'], transform)
    
    print(f"Number of classes in dataset: {len(dataset.classes)}")
    print("Class names:")
    for i, cls_name in enumerate(dataset.classes):
        print(f"Class {i}: {cls_name}")
    print(f"Min label: {min(dataset.labels)}, Max label: {max(dataset.labels)}")
    if max(dataset.labels) >= config['num_classes']:
        print(f"Warning: Dataset has labels up to {max(dataset.labels)}, but model expects {config['num_classes']} classes")
        config['num_classes'] = len(dataset.classes)
        print(f"Adjusted config['num_classes'] to {config['num_classes']} to match dataset")

    if os.path.exists(SPLIT_FILE):
        train_idx, val_idx, test_idx = load_split_indices(SPLIT_FILE)
    else:
        train_idx, val_idx, test_idx = split_dataset(dataset, config['test_size'], config['val_size'], config['seed'])
        save_split_indices(train_idx, val_idx, test_idx, SPLIT_FILE)

    val_dataset = torch.utils.data.Subset(dataset, val_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

    if config['loss_type'] == 'class_balanced':
        train_labels = [dataset.labels[i] for i in train_idx]
        samples_per_cls = torch.zeros(config['num_classes'])
        for i, count in Counter(train_labels).items():
            samples_per_cls[i] = count
        criterion = get_loss_function(config['loss_type'], beta=config['beta'], gamma=config['gamma'])
    else:
        criterion = get_loss_function(config['loss_type'], gamma=config['gamma'])
        samples_per_cls = None

    soup_dir = 'ViT-B_Soup'
    model_paths = [os.path.join(soup_dir, f) for f in os.listdir(soup_dir) if f.endswith('.pth')]
    if model_paths:
        print(f"Found {len(model_paths)} models in {soup_dir}: {model_paths}")
        if config['soup_type'] == 'greedy':
            perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'uniform':
            perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'weighted':
            perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'diversity':
            perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'prediction':
            perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config)
    else:
        print(f"No .pth files found in {soup_dir}. Skipping greedy soup.")

config = {
    'seed': 10,
    'model_name': "google/vit-base-patch16-224-in21k",
    'model_local_path': "./local_model/vit-base-patch16-224-in21k",
    'num_classes': 257,  
    'dataset_path': os.path.join(kagglehub.dataset_download("jessicali9530/caltech256"), "256_ObjectCategories"),
    'test_size': 0.3,
    'val_size': 0.5,
    'augmentation_params': {},
    'batch_size': 64,
    'num_workers': 0,
    'lr': 1e-4,
    'T_max': 25,
    'soup_type':'weighted',
    'loss_type': 'focal',
    'beta': 0.9999,
    'gamma': 2.0,
    'num_epochs': 25,
    'patience': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'load_checkpoint': True,
    'freeze_layers': 6,
}

if __name__ == "__main__":
    main(config)

Loading processor from local folder: ./local_model/vit-base-patch16-224-in21k
Number of classes in dataset: 257
Class names:
Class 0: 001.ak47
Class 1: 002.american-flag
Class 2: 003.backpack
Class 3: 004.baseball-bat
Class 4: 005.baseball-glove
Class 5: 006.basketball-hoop
Class 6: 007.bat
Class 7: 008.bathtub
Class 8: 009.bear
Class 9: 010.beer-mug
Class 10: 011.billiards
Class 11: 012.binoculars
Class 12: 013.birdbath
Class 13: 014.blimp
Class 14: 015.bonsai-101
Class 15: 016.boom-box
Class 16: 017.bowling-ball
Class 17: 018.bowling-pin
Class 18: 019.boxing-glove
Class 19: 020.brain-101
Class 20: 021.breadmaker
Class 21: 022.buddha-101
Class 22: 023.bulldozer
Class 23: 024.butterfly
Class 24: 025.cactus
Class 25: 026.cake
Class 26: 027.calculator
Class 27: 028.camel
Class 28: 029.cannon
Class 29: 030.canoe
Class 30: 031.car-tire
Class 31: 032.cartman
Class 32: 033.cd
Class 33: 034.centipede
Class 34: 035.cereal-box
Class 35: 036.chandelier-101
Class 36: 037.chess-board
Class 37: 038

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./local_model/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([257]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([257, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Weighted soup saved to ./Soup_Saved_5_ViTB\weighted_soup.pth
Weighted Soup Test Results: Loss: 1.0161, Accuracy: 0.9310, Macro F1: 0.9359


# Uniform Soup

In [ ]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
from collections import Counter
from torch.nn import functional as F
import kagglehub
import time
import logging
import matplotlib.pyplot as plt
from datetime import datetime
import re
import csv

CHECKPOINT_DIR = './Soup_Saved_5_ViTB'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SPLIT_FILE = "split_indices.pkl"

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ClassBalancedLoss(torch.nn.Module):
    def __init__(self, beta, gamma, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, samples_per_cls):
        num_classes = logits.size(1)
        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        weights = weights.to(logits.device)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1 - probs, self.gamma)
        cb_loss = -weights.unsqueeze(0) * focal_weight * targets_one_hot * log_probs
        loss = cb_loss.sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

class Caltech256Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.images = []
        self.labels = []
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, img_name))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": torch.tensor(label, dtype=torch.long)}

def load_dataset(root_dir, transform):
    return Caltech256Dataset(root_dir=root_dir, transform=transform)

def split_dataset(dataset, test_size=0.3, val_size=0.5, random_state=11):
    labels = dataset.labels
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))
    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    val_idx, test_idx = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))
    val_idx = [temp_idx[i] for i in val_idx]
    test_idx = [temp_idx[i] for i in test_idx]
    return train_idx, val_idx, test_idx

def save_split_indices(train_idx, val_idx, test_idx, file_path=SPLIT_FILE):
    with open(file_path, "wb") as f:
        pickle.dump((train_idx, val_idx, test_idx), f)
    print(f"Split indices saved to {file_path}")

def load_split_indices(file_path=SPLIT_FILE):
    with open(file_path, "rb") as f:
        train_idx, val_idx, test_idx = pickle.load(f)
    print(f"Loaded split indices from {file_path}")
    return train_idx, val_idx, test_idx

def get_transform(processor, aug_params=None):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])

def get_model(model_name, num_classes, freeze_layers=0, local_model_path=None):
    if local_model_path and os.path.exists(local_model_path):
        print(f"Loading model from local folder: {local_model_path}")
        model = ViTForImageClassification.from_pretrained(
            local_model_path,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        print("Downloading model from Hugging Face...")
        model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        if local_model_path:
            os.makedirs(local_model_path, exist_ok=True)
            model.save_pretrained(local_model_path)
            print(f"Model saved to {local_model_path}")
    if freeze_layers > 0:
        num_layers = len(model.vit.encoder.layer)
        freeze_layers = min(freeze_layers, num_layers)
        for i in range(freeze_layers):
            for param in model.vit.encoder.layer[i].parameters():
                param.requires_grad = False
        print(f"Froze the first {freeze_layers} layers of the ViT encoder.")
    return model

def get_loss_function(loss_type, **kwargs):
    if loss_type == 'focal':
        return FocalLoss(gamma=kwargs.get('gamma', 2.0))
    elif loss_type == 'class_balanced':
        return ClassBalancedLoss(beta=kwargs.get('beta', 0.9999), gamma=kwargs.get('gamma', 2.0))
    else:
        raise ValueError("unsupported loss type")

def evaluate_model(model, data_loader, criterion, device, samples_per_cls=None):
    model.eval()
    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            if labels_batch.max() >= model.classifier.out_features:
                raise ValueError(f"Label {labels_batch.max().item()} exceeds model classes {model.classifier.out_features}")
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values)
                logits = outputs.logits
                loss = criterion(logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    return avg_loss, accuracy, macro_f1
def perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    uniform_soup_state_dict = average_state_dicts(state_dicts)
    
    model.load_state_dict(uniform_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'uniform_soup.pth')
    torch.save({'model_state_dict': uniform_soup_state_dict}, soup_checkpoint_path)
    print(f"Uniform soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Uniform Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    accuracies = []
    for i, state_dict in enumerate(state_dicts):
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        accuracies.append(acc)
        model.to('cpu')
        torch.cuda.empty_cache()

    weights = torch.softmax(torch.tensor(accuracies), dim=0).tolist()
    weighted_soup_state_dict = {key: sum(w * sd[key] for w, sd in zip(weights, state_dicts)) for key in state_dicts[0].keys()}

    model.load_state_dict(weighted_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'weighted_soup.pth')
    torch.save({'model_state_dict': weighted_soup_state_dict}, soup_checkpoint_path)
    print(f"Weighted soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Weighted Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def compute_cosine_similarity(state_dict1, state_dict2):
    flat1 = torch.cat([p.flatten() for p in state_dict1.values()])
    flat2 = torch.cat([p.flatten() for p in state_dict2.values()])
    return F.cosine_similarity(flat1, flat2, dim=0).item()

def perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config, k=3):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    selected_indices = [0]
    for _ in range(k - 1):
        max_diversity = -1
        best_idx = None
        for i in range(len(state_dicts)):
            if i not in selected_indices:
                avg_similarity = sum(compute_cosine_similarity(state_dicts[i], state_dicts[j]) for j in selected_indices) / len(selected_indices)
                diversity = 1 - avg_similarity
                if diversity > max_diversity:
                    max_diversity = diversity
                    best_idx = i
        selected_indices.append(best_idx)

    diverse_state_dicts = [state_dicts[i] for i in selected_indices]
    diversity_soup_state_dict = average_state_dicts(diverse_state_dicts)
    
    model.load_state_dict(diversity_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'diversity_soup.pth')
    torch.save({'model_state_dict': diversity_soup_state_dict}, soup_checkpoint_path)
    print(f"Diversity soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Diversity Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            avg_logits = torch.zeros(len(labels_batch), num_classes, device=device)
            for path in model_paths:
                model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
                state_dict = torch.load(path, map_location='cpu')['model_state_dict']
                model.load_state_dict(state_dict)
                model.to(device)
                model.eval()
                outputs = model(pixel_values)
                avg_logits += outputs.logits / len(model_paths)
                model.to('cpu')   
                torch.cuda.empty_cache()  
            loss = criterion(avg_logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(avg_logits, labels_batch)
            total_loss += loss.item()
            preds.extend(avg_logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    print(f"Prediction Soup Test Results: Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Macro F1: {macro_f1:.4f}")


def perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    cpu_device = torch.device('cpu')
    num_classes = config['num_classes']  
    model_name = config['model_name']
    local_model_path = config['model_local_path']
    
   
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    
    state_dicts = []
    for path in model_paths:
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint['model_state_dict']
        checkpoint_classes = state_dict['classifier.weight'].shape[0]
        if checkpoint_classes != num_classes:
            print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
            if checkpoint_classes < num_classes:
                padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                padding_bias = torch.zeros(num_classes - checkpoint_classes)
                state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
            elif checkpoint_classes > num_classes:
                state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
        state_dicts.append(state_dict)
    
    individual_accuracies = []
    for i, state_dict in enumerate(state_dicts):
        print(f"Loading and evaluating model {i} from {model_paths[i]}")
        model.load_state_dict(state_dict)
        model.to(device) 
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        individual_accuracies.append((i, acc))
        print(f"Model {i}: Validation Accuracy = {acc:.4f}")
        model.to(cpu_device)
        torch.cuda.empty_cache()  
    
    sorted_models = sorted(individual_accuracies, key=lambda x: x[1], reverse=True)
    print("\nSorted models by validation accuracy:")
    for idx, acc in sorted_models:
        print(f"Model {idx}: {acc:.4f}")
    
    best_idx, best_acc = sorted_models[0]
    soup_indices = [best_idx]
    current_soup_acc = best_acc
    print(f"\nInitial soup with model {best_idx}, accuracy: {best_acc:.4f}")
    
    for idx, acc in sorted_models[1:]:
        temp_soup_indices = soup_indices + [idx]
        temp_soup_state_dict = average_state_dicts([state_dicts[i] for i in temp_soup_indices])
        model.load_state_dict(temp_soup_state_dict)
        model.to(device) 
        print(f"Evaluating temporary soup with models {temp_soup_indices}")
        _, temp_acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        model.to(cpu_device)
        torch.cuda.empty_cache()  
        if temp_acc > current_soup_acc:
            soup_indices.append(idx)
            current_soup_acc = temp_acc
            print(f"Added model {idx}, new soup accuracy: {temp_acc:.4f}")
        else:
            print(f"Model {idx} did not improve soup, skipping (accuracy: {temp_acc:.4f})")
    
    final_soup_state_dict = average_state_dicts([state_dicts[i] for i in soup_indices])
    model.load_state_dict(final_soup_state_dict)
    model.to(device) 
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'greedy_soup.pth')
    torch.save({'model_state_dict': final_soup_state_dict}, soup_checkpoint_path)
    print(f"Final soup saved to {soup_checkpoint_path}")
    
    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to(cpu_device)
    torch.cuda.empty_cache()  
    print(f"\nFinal Soup Test Results:")
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")

def average_state_dicts(state_dicts):
    if not state_dicts:
        raise ValueError("No state_dicts to average")
    soup_state_dict = {}
    for key in state_dicts[0].keys():
        soup_state_dict[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return soup_state_dict

def main(config):
    torch.manual_seed(config['seed'])
    np.random.seed(config['seed'])
    torch.backends.cudnn.deterministic = True

    if os.path.exists(config['model_local_path']):
        print(f"Loading processor from local folder: {config['model_local_path']}")
        processor = ViTImageProcessor.from_pretrained(config['model_local_path'])
    else:
        print("Downloading processor from Hugging Face...")
        processor = ViTImageProcessor.from_pretrained(config['model_name'])
        os.makedirs(config['model_local_path'], exist_ok=True)
        processor.save_pretrained(config['model_local_path'])
        print(f"Processor saved to {config['model_local_path']}")

    transform = get_transform(processor)
    dataset = load_dataset(config['dataset_path'], transform)
    
    print(f"Number of classes in dataset: {len(dataset.classes)}")
    print("Class names:")
    for i, cls_name in enumerate(dataset.classes):
        print(f"Class {i}: {cls_name}")
    print(f"Min label: {min(dataset.labels)}, Max label: {max(dataset.labels)}")
    if max(dataset.labels) >= config['num_classes']:
        print(f"Warning: Dataset has labels up to {max(dataset.labels)}, but model expects {config['num_classes']} classes")
        config['num_classes'] = len(dataset.classes)
        print(f"Adjusted config['num_classes'] to {config['num_classes']} to match dataset")

    if os.path.exists(SPLIT_FILE):
        train_idx, val_idx, test_idx = load_split_indices(SPLIT_FILE)
    else:
        train_idx, val_idx, test_idx = split_dataset(dataset, config['test_size'], config['val_size'], config['seed'])
        save_split_indices(train_idx, val_idx, test_idx, SPLIT_FILE)

    val_dataset = torch.utils.data.Subset(dataset, val_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

    if config['loss_type'] == 'class_balanced':
        train_labels = [dataset.labels[i] for i in train_idx]
        samples_per_cls = torch.zeros(config['num_classes'])
        for i, count in Counter(train_labels).items():
            samples_per_cls[i] = count
        criterion = get_loss_function(config['loss_type'], beta=config['beta'], gamma=config['gamma'])
    else:
        criterion = get_loss_function(config['loss_type'], gamma=config['gamma'])
        samples_per_cls = None

    soup_dir = 'ViT-B_Soup'
    model_paths = [os.path.join(soup_dir, f) for f in os.listdir(soup_dir) if f.endswith('.pth')]
    if model_paths:
        print(f"Found {len(model_paths)} models in {soup_dir}: {model_paths}")
        if config['soup_type'] == 'greedy':
            perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'uniform':
            perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'weighted':
            perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'diversity':
            perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'prediction':
            perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config)
    else:
        print(f"No .pth files found in {soup_dir}. Skipping greedy soup.")

config = {
    'seed': 10,
    'model_name': "google/vit-base-patch16-224-in21k",
    'model_local_path': "./local_model/vit-base-patch16-224-in21k",
    'num_classes': 257,
    'dataset_path': os.path.join(kagglehub.dataset_download("jessicali9530/caltech256"), "256_ObjectCategories"),
    'test_size': 0.3,
    'val_size': 0.5,
    'augmentation_params': {},
    'batch_size': 64,
    'num_workers': 0,
    'lr': 1e-4,
    'T_max': 25,
    'soup_type':'uniform',
    'loss_type': 'focal',
    'beta': 0.9999,
    'gamma': 2.0,
    'num_epochs': 25,
    'patience': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'load_checkpoint': True,
    'freeze_layers': 6,
}

if __name__ == "__main__":
    main(config)

c:\Users\Dell-G5\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading processor from local folder: ./local_model/vit-base-patch16-224-in21k
Number of classes in dataset: 257
Class names:
Class 0: 001.ak47
Class 1: 002.american-flag
Class 2: 003.backpack
Class 3: 004.baseball-bat
Class 4: 005.baseball-glove
Class 5: 006.basketball-hoop
Class 6: 007.bat
Class 7: 008.bathtub
Class 8: 009.bear
Class 9: 010.beer-mug
Class 10: 011.billiards
Class 11: 012.binoculars
Class 12: 013.birdbath
Class 13: 014.blimp
Class 14: 015.bonsai-101
Class 15: 016.boom-box
Class 16: 017.bowling-ball
Class 17: 018.bowling-pin
Class 18: 019.boxing-glove
Class 19: 020.brain-101
Class 20: 021.breadmaker
Class 21: 022.buddha-101
Class 22: 023.bulldozer
Class 23: 024.butterfly
Class 24: 025.cactus
Class 25: 026.cake
Class 26: 027.calculator
Class 27: 028.camel
Class 28: 029.cannon
Class 29: 030.canoe
Class 30: 031.car-tire
Class 31: 032.cartman
Class 32: 033.cd
Class 33: 034.centipede
Class 34: 035.cereal-box
Class 35: 036.chandelier-101
Class 36: 037.chess-board
Class 37: 038

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./local_model/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([257]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([257, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Uniform soup saved to ./Soup_Saved_5_ViTB\uniform_soup.pth
Uniform Soup Test Results: Loss: 1.0400, Accuracy: 0.9297, Macro F1: 0.9350


# Greedy Soup

In [ ]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
from collections import Counter
from torch.nn import functional as F
import kagglehub
import time
import logging
import matplotlib.pyplot as plt
from datetime import datetime
import re
import csv

CHECKPOINT_DIR = './Soup_Saved_5_ViTB'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SPLIT_FILE = "split_indices.pkl"

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class ClassBalancedLoss(torch.nn.Module):
    def __init__(self, beta, gamma, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets, samples_per_cls):
        num_classes = logits.size(1)
        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * num_classes
        weights = weights.to(logits.device)
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1 - probs, self.gamma)
        cb_loss = -weights.unsqueeze(0) * focal_weight * targets_one_hot * log_probs
        loss = cb_loss.sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

class Caltech256Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.images = []
        self.labels = []
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(cls_dir, img_name))
                        self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": torch.tensor(label, dtype=torch.long)}

def load_dataset(root_dir, transform):
    return Caltech256Dataset(root_dir=root_dir, transform=transform)

def split_dataset(dataset, test_size=0.3, val_size=0.5, random_state=11):
    labels = dataset.labels
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(np.zeros(len(labels)), labels))
    temp_labels = [labels[i] for i in temp_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    val_idx, test_idx = next(sss2.split(np.zeros(len(temp_labels)), temp_labels))
    val_idx = [temp_idx[i] for i in val_idx]
    test_idx = [temp_idx[i] for i in test_idx]
    return train_idx, val_idx, test_idx

def save_split_indices(train_idx, val_idx, test_idx, file_path=SPLIT_FILE):
    with open(file_path, "wb") as f:
        pickle.dump((train_idx, val_idx, test_idx), f)
    print(f"Split indices saved to {file_path}")

def load_split_indices(file_path=SPLIT_FILE):
    with open(file_path, "rb") as f:
        train_idx, val_idx, test_idx = pickle.load(f)
    print(f"Loaded split indices from {file_path}")
    return train_idx, val_idx, test_idx

def get_transform(processor, aug_params=None):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
    ])

def get_model(model_name, num_classes, freeze_layers=0, local_model_path=None):
    if local_model_path and os.path.exists(local_model_path):
        print(f"Loading model from local folder: {local_model_path}")
        model = ViTForImageClassification.from_pretrained(
            local_model_path,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        print("Downloading model from Hugging Face...")
        model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        if local_model_path:
            os.makedirs(local_model_path, exist_ok=True)
            model.save_pretrained(local_model_path)
            print(f"Model saved to {local_model_path}")
    if freeze_layers > 0:
        num_layers = len(model.vit.encoder.layer)
        freeze_layers = min(freeze_layers, num_layers)
        for i in range(freeze_layers):
            for param in model.vit.encoder.layer[i].parameters():
                param.requires_grad = False
        print(f"Froze the first {freeze_layers} layers of the ViT encoder.")
    return model

def get_loss_function(loss_type, **kwargs):
    if loss_type == 'focal':
        return FocalLoss(gamma=kwargs.get('gamma', 2.0))
    elif loss_type == 'class_balanced':
        return ClassBalancedLoss(beta=kwargs.get('beta', 0.9999), gamma=kwargs.get('gamma', 2.0))
    else:
        raise ValueError("unsupported loss type")

def evaluate_model(model, data_loader, criterion, device, samples_per_cls=None):
    model.eval()
    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            if labels_batch.max() >= model.classifier.out_features:
                raise ValueError(f"Label {labels_batch.max().item()} exceeds model classes {model.classifier.out_features}")
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values)
                logits = outputs.logits
                loss = criterion(logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    return avg_loss, accuracy, macro_f1
def perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    uniform_soup_state_dict = average_state_dicts(state_dicts)
    
    model.load_state_dict(uniform_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'uniform_soup.pth')
    torch.save({'model_state_dict': uniform_soup_state_dict}, soup_checkpoint_path)
    print(f"Uniform soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Uniform Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    accuracies = []
    for i, state_dict in enumerate(state_dicts):
        model.load_state_dict(state_dict)
        model.to(device)
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        accuracies.append(acc)
        model.to('cpu')
        torch.cuda.empty_cache()

    weights = torch.softmax(torch.tensor(accuracies), dim=0).tolist()
    weighted_soup_state_dict = {key: sum(w * sd[key] for w, sd in zip(weights, state_dicts)) for key in state_dicts[0].keys()}

    model.load_state_dict(weighted_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'weighted_soup.pth')
    torch.save({'model_state_dict': weighted_soup_state_dict}, soup_checkpoint_path)
    print(f"Weighted soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Weighted Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def compute_cosine_similarity(state_dict1, state_dict2):
    flat1 = torch.cat([p.flatten() for p in state_dict1.values()])
    flat2 = torch.cat([p.flatten() for p in state_dict2.values()])
    return F.cosine_similarity(flat1, flat2, dim=0).item()

def perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config, k=3):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    state_dicts = [torch.load(path, map_location='cpu')['model_state_dict'] for path in model_paths]
    
    selected_indices = [0]
    for _ in range(k - 1):
        max_diversity = -1
        best_idx = None
        for i in range(len(state_dicts)):
            if i not in selected_indices:
                avg_similarity = sum(compute_cosine_similarity(state_dicts[i], state_dicts[j]) for j in selected_indices) / len(selected_indices)
                diversity = 1 - avg_similarity
                if diversity > max_diversity:
                    max_diversity = diversity
                    best_idx = i
        selected_indices.append(best_idx)

    diverse_state_dicts = [state_dicts[i] for i in selected_indices]
    diversity_soup_state_dict = average_state_dicts(diverse_state_dicts)
    
    model.load_state_dict(diversity_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'diversity_soup.pth')
    torch.save({'model_state_dict': diversity_soup_state_dict}, soup_checkpoint_path)
    print(f"Diversity soup saved to {soup_checkpoint_path}")

    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to('cpu')   
    torch.cuda.empty_cache()  
    print(f"Diversity Soup Test Results: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")


def perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    num_classes = config['num_classes']
    model_name = config['model_name']
    local_model_path = config['model_local_path']

    total_loss = 0
    preds = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)
            avg_logits = torch.zeros(len(labels_batch), num_classes, device=device)
            for path in model_paths:
                model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
                state_dict = torch.load(path, map_location='cpu')['model_state_dict']
                model.load_state_dict(state_dict)
                model.to(device)
                model.eval()
                outputs = model(pixel_values)
                avg_logits += outputs.logits / len(model_paths)
                model.to('cpu')   
                torch.cuda.empty_cache()  
            loss = criterion(avg_logits, labels_batch, samples_per_cls) if isinstance(criterion, ClassBalancedLoss) else criterion(avg_logits, labels_batch)
            total_loss += loss.item()
            preds.extend(avg_logits.argmax(dim=-1).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, preds)
    macro_f1 = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)[2]
    print(f"Prediction Soup Test Results: Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}, Macro F1: {macro_f1:.4f}")


def perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config):
    device = config['device']
    cpu_device = torch.device('cpu')
    num_classes = config['num_classes']  
    model_name = config['model_name']
    local_model_path = config['model_local_path']
    
   
    model = get_model(model_name, num_classes, freeze_layers=0, local_model_path=local_model_path)
    
    state_dicts = []
    for path in model_paths:
        checkpoint = torch.load(path, map_location='cpu')
        state_dict = checkpoint['model_state_dict']
        checkpoint_classes = state_dict['classifier.weight'].shape[0]
        if checkpoint_classes != num_classes:
            print(f"Adjusting classifier from {checkpoint_classes} to {num_classes} classes for model {path}")
            if checkpoint_classes < num_classes:
                padding_weight = torch.zeros(num_classes - checkpoint_classes, state_dict['classifier.weight'].shape[1])
                padding_bias = torch.zeros(num_classes - checkpoint_classes)
                state_dict['classifier.weight'] = torch.cat([state_dict['classifier.weight'], padding_weight], dim=0)
                state_dict['classifier.bias'] = torch.cat([state_dict['classifier.bias'], padding_bias], dim=0)
            elif checkpoint_classes > num_classes:
                state_dict['classifier.weight'] = state_dict['classifier.weight'][:num_classes, :]
                state_dict['classifier.bias'] = state_dict['classifier.bias'][:num_classes]
        state_dicts.append(state_dict)
    
    individual_accuracies = []
    for i, state_dict in enumerate(state_dicts):
        print(f"Loading and evaluating model {i} from {model_paths[i]}")
        model.load_state_dict(state_dict)
        model.to(device) 
        _, acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        individual_accuracies.append((i, acc))
        print(f"Model {i}: Validation Accuracy = {acc:.4f}")
        model.to(cpu_device)
        torch.cuda.empty_cache()  
    
    sorted_models = sorted(individual_accuracies, key=lambda x: x[1], reverse=True)
    print("\nSorted models by validation accuracy:")
    for idx, acc in sorted_models:
        print(f"Model {idx}: {acc:.4f}")
    
    best_idx, best_acc = sorted_models[0]
    soup_indices = [best_idx]
    current_soup_acc = best_acc
    print(f"\nInitial soup with model {best_idx}, accuracy: {best_acc:.4f}")
    
    for idx, acc in sorted_models[1:]:
        temp_soup_indices = soup_indices + [idx]
        temp_soup_state_dict = average_state_dicts([state_dicts[i] for i in temp_soup_indices])
        model.load_state_dict(temp_soup_state_dict)
        model.to(device) 
        print(f"Evaluating temporary soup with models {temp_soup_indices}")
        _, temp_acc, _ = evaluate_model(model, val_loader, criterion, device, samples_per_cls)
        model.to(cpu_device)
        torch.cuda.empty_cache()  
        if temp_acc > current_soup_acc:
            soup_indices.append(idx)
            current_soup_acc = temp_acc
            print(f"Added model {idx}, new soup accuracy: {temp_acc:.4f}")
        else:
            print(f"Model {idx} did not improve soup, skipping (accuracy: {temp_acc:.4f})")
    
    final_soup_state_dict = average_state_dicts([state_dicts[i] for i in soup_indices])
    model.load_state_dict(final_soup_state_dict)
    model.to(device)
    soup_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'greedy_soup.pth')
    torch.save({'model_state_dict': final_soup_state_dict}, soup_checkpoint_path)
    print(f"Final soup saved to {soup_checkpoint_path}")
    
    test_loss, test_accuracy, test_macro_f1 = evaluate_model(model, test_loader, criterion, device, samples_per_cls)
    model.to(cpu_device)
    torch.cuda.empty_cache()  
    print(f"\nFinal Soup Test Results:")
    print(f"Test Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Macro F1: {test_macro_f1:.4f}")

def average_state_dicts(state_dicts):
    if not state_dicts:
        raise ValueError("No state_dicts to average")
    soup_state_dict = {}
    for key in state_dicts[0].keys():
        soup_state_dict[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return soup_state_dict

def main(config):
    torch.manual_seed(config['seed'])
    np.random.seed(config['seed'])
    torch.backends.cudnn.deterministic = True

    if os.path.exists(config['model_local_path']):
        print(f"Loading processor from local folder: {config['model_local_path']}")
        processor = ViTImageProcessor.from_pretrained(config['model_local_path'])
    else:
        print("Downloading processor from Hugging Face...")
        processor = ViTImageProcessor.from_pretrained(config['model_name'])
        os.makedirs(config['model_local_path'], exist_ok=True)
        processor.save_pretrained(config['model_local_path'])
        print(f"Processor saved to {config['model_local_path']}")

    transform = get_transform(processor)
    dataset = load_dataset(config['dataset_path'], transform)
    
    print(f"Number of classes in dataset: {len(dataset.classes)}")
    print("Class names:")
    for i, cls_name in enumerate(dataset.classes):
        print(f"Class {i}: {cls_name}")
    print(f"Min label: {min(dataset.labels)}, Max label: {max(dataset.labels)}")
    if max(dataset.labels) >= config['num_classes']:
        print(f"Warning: Dataset has labels up to {max(dataset.labels)}, but model expects {config['num_classes']} classes")
        config['num_classes'] = len(dataset.classes)
        print(f"Adjusted config['num_classes'] to {config['num_classes']} to match dataset")

    if os.path.exists(SPLIT_FILE):
        train_idx, val_idx, test_idx = load_split_indices(SPLIT_FILE)
    else:
        train_idx, val_idx, test_idx = split_dataset(dataset, config['test_size'], config['val_size'], config['seed'])
        save_split_indices(train_idx, val_idx, test_idx, SPLIT_FILE)

    val_dataset = torch.utils.data.Subset(dataset, val_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

    if config['loss_type'] == 'class_balanced':
        train_labels = [dataset.labels[i] for i in train_idx]
        samples_per_cls = torch.zeros(config['num_classes'])
        for i, count in Counter(train_labels).items():
            samples_per_cls[i] = count
        criterion = get_loss_function(config['loss_type'], beta=config['beta'], gamma=config['gamma'])
    else:
        criterion = get_loss_function(config['loss_type'], gamma=config['gamma'])
        samples_per_cls = None

    soup_dir = 'ViT-B_Soup'
    model_paths = [os.path.join(soup_dir, f) for f in os.listdir(soup_dir) if f.endswith('.pth')]
    if model_paths:
        print(f"Found {len(model_paths)} models in {soup_dir}: {model_paths}")
        if config['soup_type'] == 'greedy':
            perform_greedy_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'uniform':
            perform_uniform_soup(model_paths, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'weighted':
            perform_weighted_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'diversity':
            perform_diversity_soup(model_paths, val_loader, test_loader, criterion, samples_per_cls, config)
        elif config['soup_type'] == 'prediction':
            perform_prediction_soup(model_paths, test_loader, criterion, samples_per_cls, config)
    else:
        print(f"No .pth files found in {soup_dir}. Skipping greedy soup.")

config = {
    'seed': 10,
    'model_name': "google/vit-base-patch16-224-in21k",
    'model_local_path': "./local_model/vit-base-patch16-224-in21k",
    'num_classes': 257,
    'dataset_path': os.path.join(kagglehub.dataset_download("jessicali9530/caltech256"), "256_ObjectCategories"),
    'test_size': 0.3,
    'val_size': 0.5,
    'augmentation_params': {},
    'batch_size': 64,
    'num_workers': 0,
    'lr': 1e-4,
    'T_max': 25,
    'soup_type':'greedy',
    'loss_type': 'focal',
    'beta': 0.9999,
    'gamma': 2.0,
    'num_epochs': 25,
    'patience': 5,
    'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'load_checkpoint': True,
    'freeze_layers': 6,
}

if __name__ == "__main__":
    main(config)

c:\Users\Dell-G5\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading processor from local folder: ./local_model/vit-base-patch16-224-in21k
Number of classes in dataset: 257
Class names:
Class 0: 001.ak47
Class 1: 002.american-flag
Class 2: 003.backpack
Class 3: 004.baseball-bat
Class 4: 005.baseball-glove
Class 5: 006.basketball-hoop
Class 6: 007.bat
Class 7: 008.bathtub
Class 8: 009.bear
Class 9: 010.beer-mug
Class 10: 011.billiards
Class 11: 012.binoculars
Class 12: 013.birdbath
Class 13: 014.blimp
Class 14: 015.bonsai-101
Class 15: 016.boom-box
Class 16: 017.bowling-ball
Class 17: 018.bowling-pin
Class 18: 019.boxing-glove
Class 19: 020.brain-101
Class 20: 021.breadmaker
Class 21: 022.buddha-101
Class 22: 023.bulldozer
Class 23: 024.butterfly
Class 24: 025.cactus
Class 25: 026.cake
Class 26: 027.calculator
Class 27: 028.camel
Class 28: 029.cannon
Class 29: 030.canoe
Class 30: 031.car-tire
Class 31: 032.cartman
Class 32: 033.cd
Class 33: 034.centipede
Class 34: 035.cereal-box
Class 35: 036.chandelier-101
Class 36: 037.chess-board
Class 37: 038

Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./local_model/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([257]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([257, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading and evaluating model 0 from ViT-B_Soup\ViT-B1.pth
Model 0: Validation Accuracy = 0.9268
Loading and evaluating model 1 from ViT-B_Soup\ViT-B2.pth
Model 1: Validation Accuracy = 0.8785
Loading and evaluating model 2 from ViT-B_Soup\ViT-B3.pth
Model 2: Validation Accuracy = 0.9076
Loading and evaluating model 3 from ViT-B_Soup\ViT-B4.pth
Model 3: Validation Accuracy = 0.9216
Loading and evaluating model 4 from ViT-B_Soup\ViT-B5.pth
Model 4: Validation Accuracy = 0.8469

Sorted models by validation accuracy:
Model 0: 0.9268
Model 3: 0.9216
Model 2: 0.9076
Model 1: 0.8785
Model 4: 0.8469

Initial soup with model 0, accuracy: 0.9268
Evaluating temporary soup with models [0, 3]
Added model 3, new soup accuracy: 0.9314
Evaluating temporary soup with models [0, 3, 2]
Model 2 did not improve soup, skipping (accuracy: 0.9262)
Evaluating temporary soup with models [0, 3, 1]
Model 1 did not improve soup, skipping (accuracy: 0.9288)
Evaluating temporary soup with models [0, 3, 4]
Model 4 di